##### Traitement de données pour EDF


In [54]:
import pandas as pd
import numpy as np


export_df = pd.read_csv('datasets example/export.csv', sep=';', dtype=str)
result_df = pd.read_csv('datasets example/result.csv', sep=';', dtype=str)
nomenclature_df = pd.read_csv('Nomenclature_ICPE.csv', sep=';', dtype=str)


In [55]:
export_df.head()

,Numéro d'établissement,Nom établissement,Adresse 1,Adresse 2,Adresse 3,Code postal,Commune,Régime en vigueur,Statut SEVESO,Date de dernière inspection,Statut IED,Numéro SIRET
0,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,NaN,NaN,33240,Gauriaguet,Autorisation,Non Seveso,NaN,non,42298971500186
1,0000000004,SOCIETE FERROLAC,4 RUE PIERRE GILLES DE GENNES,ZAC DE LA VIGONNIERE,NaN,18400,Saint-Florent-sur-Cher,Autorisation,Non Seveso,2023-05-16,non,78560962900064
2,0000000005,ORLEANS METROPOLE - Parc de Loire,5 PL DU 6 JUIN 1944,ESPACE SAINT MARC,45058 ORLEANS CEDEX 1,45000,Orléans,Autorisation,Non Seveso,NaN,non,24450046800040
3,0000000010,GRANUPLAST FRANCE,754 rue de la liberté,ZI la grande borne,NaN,01480,Jassans-Riottier,Non ICPE,NaN,2024-10-23,non,88325431000021
4,0000000034,EOLIEN - TOTAL QUADRAN CE La Luçoise,Forêt de Chaniaux,Lou Chambon,NaN,48250,Luc,Autorisation,Non Seveso,NaN,non,81519422000027


In [56]:
result_df.head()

,code_aiot,x,y,code_epsg,nom_ets,num_dep,adresse,cd_insee,cd_postal,commune,...,eolienne,industrie,ied,priorite_nationale,rubriques_autorisation,rubriques_enregistrement,rubriques_declaration,date_modification,derniere_inspection,url_fiche
0,0000000002,432849,6444403,2154,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,33,Secteur de la Marquette,33183,33240,Gauriaguet,...,0,1,0,0,1450|1510|1630|4801,NaN,1436|2171|2910|2925|4110|4120|4130|4140|4150|4...,2026/05/06 00:00:00,NaN,https://www.georisques.gouv.fr/risques/install...
1,0000000004,643988,6656652,2154,SOCIETE FERROLAC,18,4 RUE PIERRE GILLES DE GENNES - ZAC DE LA VIGO...,18207,18400,Saint-Florent-sur-Cher,...,0,1,0,1,2718,2712|2713,2711,2026/02/25 00:00:00,2023/05/16 00:00:00,https://www.georisques.gouv.fr/risques/install...
2,0000000005,621725,6755411,2154,ORLEANS METROPOLE - Parc de Loire,45,5 PL DU 6 JUIN 1944 - ESPACE SAINT MARC - 4505...,45274,45000,Orléans,...,0,1,0,0,NaN,NaN,NaN,2025/09/10 00:00:00,NaN,https://www.georisques.gouv.fr/risques/install...
3,0000000010,836747,6543731,2154,GRANUPLAST FRANCE,01,754 rue de la liberté - ZI la grande borne,01194,01480,Jassans-Riottier,...,0,0,0,0,NaN,NaN,NaN,2025/08/08 00:00:00,2024/10/23 00:00:00,https://www.georisques.gouv.fr/risques/install...
4,0000000034,768626,6388530,2154,EOLIEN - TOTAL QUADRAN CE La Luçoise,48,Forêt de Chaniaux - Lou Chambon,48086,48250,Luc,...,1,0,0,0,2980,NaN,NaN,2026/02/20 00:00:00,NaN,https://www.georisques.gouv.fr/risques/install...


In [57]:
# On drop toutes les colonnes qui ne nous intéressent pas
result_df = result_df.drop(["bovins", "porcs", "volailles", "carriere", "eolienne", "industrie", "url_fiche", "code_epsg", "ied", "priorite_nationale", "num_dep"], axis =1)


In [58]:
# On renomme les colonnes liées à l'ICPE
result_df = result_df.rename({"rubriques_autorisation" : "ICPE_A" , "rubriques_enregistrement" : "ICPE_E", "rubriques_declaration" : "ICPE_D"}, axis = 1)
result_df.columns


Index(['code_aiot', 'x', 'y', 'nom_ets', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime',
       'lib_regime', 'seveso', 'lib_seveso', 'ICPE_A', 'ICPE_E', 'ICPE_D',
       'date_modification', 'derniere_inspection'],
      dtype='str')

### Dédoublement des lignes par code ICPE

In [59]:
result_simple = result_df[result_df[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1)]

In [60]:
result_df["ICPE_A"] = result_df["ICPE_A"].apply( lambda x: f"{x}|0" if pd.notna(x) else x )
result_df["ICPE_E"] = result_df["ICPE_E"].apply( lambda x: f"{x}|0" if pd.notna(x) else x )

In [61]:
result_df['ICPE_A'] = result_df['ICPE_A'].str.split('|')
result_df = result_df.explode('ICPE_A')
result_df = result_df[result_df['ICPE_A'] != "nan"]

In [62]:
#On met des NaN là où il ne doit pas y avoir de valeurs
result_df.loc[((result_df['ICPE_A'] != "0") & (result_df["ICPE_A"].notna())), "ICPE_E"] = np.nan
result_df.loc[((result_df['ICPE_A'] != "0") & (result_df["ICPE_A"].notna())), "ICPE_D"] = np.nan


In [63]:
result_df['ICPE_E'] = result_df['ICPE_E'].str.split('|')
result_df = result_df.explode('ICPE_E')

In [64]:
result_df = result_df[result_df['ICPE_E'] != "nan"]

In [65]:
result_df.loc[((result_df['ICPE_E'] != "0") & (result_df["ICPE_E"] != "nan") & (result_df['ICPE_E'].notna() )), "ICPE_D"] = np.nan

In [66]:
result_df['ICPE_D'] = result_df['ICPE_D'].str.split('|')
result_df = result_df.explode('ICPE_D')

In [67]:
result_df = result_df[result_df['ICPE_D'] != "nan"]

In [68]:
result_df.loc[(result_df['ICPE_A'] == "0"), "ICPE_A"] = np.nan
result_df.loc[(result_df['ICPE_E'] == "0"), "ICPE_E"] = np.nan

In [69]:
result_df = result_df[~result_df[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1)]

In [70]:
result_trie = pd.concat([result_df, result_simple]).sort_values(by="code_aiot")

In [71]:
result_trie.tail(35)

,code_aiot,x,y,nom_ets,adresse,cd_insee,cd_postal,commune,code_naf,lib_naf,num_siret,cd_regime,lib_regime,seveso,lib_seveso,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection
134821,0100316688,845893,6303285,Les Jardins du Vigueirat,463 route de Maillane,13100,13210,Saint-Rémy-de-Provence,NaN,NaN,88208801600010,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/19 00:00:00,2026/05/26 00:00:00
134822,0100316696,872473,6257106,GONTERO - MG13,QUARTIERS DES GLACIERES Sortie n°9,13026,13220,Châteauneuf-les-Martigues,NaN,NaN,40859282200014,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/16 00:00:00,2026/06/09 00:00:00
134823,0100316710,591263,6816126,ENERCON SERVICE FRANCE,19-21 avenue Gustave Eiffel - Bât. B 10 Espace...,28177,28630,Gellainville,NaN,NaN,NaN,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/17 00:00:00,2026/06/12 00:00:00
134824,0100316762,652528,6868535,EIFFAGE ENERGIE SYSTEMES - ILE DE FRANCE,2 RUE FLORA TRISTAN,93066,93210,Saint-Denis,43,Travaux de construction spécialisés,42054064300269,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/17 00:00:00,2026/06/08 00:00:00
134825,0100316764,677032,6797373,SETA ENVIRONNEMENT,4 RUE DES CHAMPARTS,77431,77820,Le Chatelet-en-Brie,42,Génie civil,79934430400024,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/17 00:00:00,2026/06/03 00:00:00
134826,0100316767,661598,6860681,SOC NOUV TRAVAUX PUBLICS ET PARTICULIERS,2 RUE DE LA CORNEILLE,94033,94120,Fontenay-sous-Bois,42,Génie civil,57207510900015,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/13 00:00:00,2026/06/08 00:00:00
134827,0100316782,661504,6847672,SERPOLLET ILE DE FRANCE,19 RUE LE BOIS CERDON,94074,94460,Valenton,43,Travaux de construction spécialisés,84108842000036,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/23 00:00:00,2026/06/08 00:00:00
134828,0100316785,362895,7679789,SAMARAPATY CEDRIC,436 CHEMIN JEANSON - RDM LES BAS,97409,97440,Saint-Andre,45,Commerce et réparation d'automobiles et de mot...,42316714700036,E,Enregistrement,3,Non Seveso,NaN,NaN,NaN,2026/06/27 00:00:00,NaN
134829,0100316793,649485,6847885,TERIDEAL SEGEX ENERGIES,13 AVENUE JEANNE GARNERIN,91689,91320,Wissous,43,Travaux de construction spécialisés,78805646300219,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/23 00:00:00,2026/06/11 00:00:00
134830,0100316798,591914,6817605,ERG FRANCE,5 rue Blaise Pascal - Bâtiment B - Lotissement 6,28085,28000,Chartres,71,Activités d'architecture et d'ingénierie ; act...,44132040500083,NEANT,Non ICPE,NaN,NaN,NaN,NaN,NaN,2026/06/17 00:00:00,2026/06/05 00:00:00


### On ajoute au fichier les descriptions des codes ICPE correspondantes

In [72]:
#On ne garde que les codes ICPE complets et non pas les grandes catégories
nomenclature_df = nomenclature_df[nomenclature_df['Service'].notna()]

In [73]:
nomenclature_df.head(10)

,Code,Description,Service
2,1185,Gaz à effet de serre fluorés,SSEEC-BPC
5,1312,Mise en œuvre de produits explosifs à des fins...,SRT-BRIEC
8,1413,Installations de remplissage de réservoirs de ...,SRT-BRIEC
9,1414,Installations de remplissage ou de distributio...,SRT-BRIEC
10,1416,Station services (hydrogène),SRT-BRIEC
12,1421,Installation de remplissage d'aérosols inflamm...,SRT-BRIEC
14,1434,Installations de remplissage ou de distributio...,SRT-BRIEC
15,1435,Stations-service,SRT-BRIEC
16,1436,Liquides de point éclair compris entre 60°C et...,SRT-BRIEC
18,1450,Solides inflammables,SRT-BRIEC


In [74]:
# On convertit les types pour la concaténation
nomenclature_df['Code'].astype(int)
result_trie['ICPE_A'].astype('Int64')

0         1450
0         <NA>
0         <NA>
0         <NA>
0         <NA>
          ... 
134851    <NA>
134852    <NA>
134853    <NA>
134854    <NA>
134855    <NA>
Name: ICPE_A, Length: 218723, dtype: Int64

In [75]:
result_trie['ICPE'] = result_trie['ICPE_A'].fillna('') + result_trie['ICPE_E'].fillna('') +  result_trie['ICPE_D'].fillna('')

In [76]:
# On merge
result = pd.merge(result_trie, nomenclature_df, left_on='ICPE', right_on="Code", how="left")


In [77]:
result=result.drop(columns=['ICPE', 'Code', 'Service'])

In [78]:
# On change l'ordre des colonnes pour plus de lisibilité
result= result[['code_aiot', 'nom_ets', 'x', 'y', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime',
       'lib_regime', 'seveso', 'lib_seveso', 'Description', 'ICPE_A', 'ICPE_E', 'ICPE_D',
       'date_modification', 'derniere_inspection']]
result= result.rename({'Description':'Description du code ICPE'}, axis = 1)
result.head()

,code_aiot,nom_ets,x,y,adresse,cd_insee,cd_postal,commune,code_naf,lib_naf,...,cd_regime,lib_regime,seveso,lib_seveso,Description du code ICPE,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection
0,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,Solides inflammables,1450,NaN,NaN,2026/05/06 00:00:00,NaN
1,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,Alcools de bouche d'origine agricole,NaN,NaN,4755,2026/05/06 00:00:00,NaN
2,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,"Liquides comburants catégorie 1, 2 ou 3",NaN,NaN,4441,2026/05/06 00:00:00,NaN
3,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,Liquides inflammables de catégorie 2 ou catégo...,NaN,NaN,4331,2026/05/06 00:00:00,NaN
4,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,Liquides inflammables de catégorie 1,NaN,NaN,4330,2026/05/06 00:00:00,NaN


In [79]:
# On récupère un excel trié
result.to_csv('result_trie.csv', index=False)